# Intro To The Z-Transform
## Chris Tralie

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd

## Motivation: Shortcomings of The DFT

The window imposes some constraints on the DFT, which can make it difficult in some practical scenarios.  We have spectral leakage, and we also need an impulse response that's exactly the same length as the input for the convolution/multiplication duality to apply.  Finally, some impulse responses are of infinite length, so there is no appropriate window to analyze them in.  We're going to need a different tool to help us with those especially


In [ ]:
sr = 44100
x = np.random.randn(3*sr)
h = np.array([1, 1]) # Simple lowpass filter
y = np.convolve(x, h)

hpad = np.zeros_like(x)
hpad[0:len(h)] = h
plt.subplot(311)
plt.plot(np.abs(np.fft.rfft(x)))
plt.title("|fft(x)|")
plt.subplot(312)
plt.plot(np.abs(np.fft.rfft(hpad)))
plt.title("|fft(h)|")
plt.subplot(313)
plt.plot(np.abs(np.fft.rfft(y[0:x.size])))
plt.title("|fft(x*h)|")
plt.tight_layout()

## Z-Transform: A Simple Example by Hand

### Ex) 
We showed in class how the following impulse response

#### $h[0] = 1, h[1] = 1$

leads to a "lowpass filter" which kills of higher frequencies. If we plug this into the equation for the convolution, we get the following equation for what the result is in terms of x, which is known as a <b>finite discrete difference equation</b>


#### $y[n] = x[n] + x[n-1]$

Now let's make 
#### $x[n] = e^{i 2 \pi f n}$ 

where $f \in [0, 0.5]$ is in units of cycles/sample and we know because of aliasing that the max f can be is 0.5. But otherwise, unlike the DFT, f can be a completely arbitrary real number, and it doesn't have to go through an integer number of cycles over any particular interval.

Now let's plug in the phasor to the difference equation:

#### $y[n] = e^{i 2 \pi f n} + e^{i 2 \pi f (n-1) }$

For convenience, let $\omega = 2 \pi f \in [0, \pi]$, frequency in radians/sample.  Then the expression turns into:

#### $y[n] = e^{i \omega n} + e^{i \omega (n-1)} = e^{i \omega n} + e^{i \omega n} e^{-i \omega}  $

I want to break this apart into an amplitude and a phase, and I want to look at the amplitude to verify that this promotes low frequencies. Use a trick where I multiply by a fancy 1, and factor some stuff out:

#### $y[n] = e^{i \omega n}(1 + e^{-i \omega}) \frac{ e^{i \omega/2} } {  e^{i \omega/2} }$

#### $y[n] = e^{i \omega n} e^{-i \omega /2} (e^{i \omega/2} + e^{-i \omega / 2})$

Now I can use an identity based on Euler's formula ($e^{ix} + e^{-ix} = 2 \cos(x)$) to change what's into the parentheses simply into a cosine

#### $y[n] = e^{i \omega n} \left( e^{-i \omega /2} (2 \cos(\omega/2)) \right)$

 When we plot this, we indeed see that it falls off for higher frequencies

In [ ]:
w = np.linspace(0, np.pi, 100)
mag = 2*np.cos(w/2)
plt.plot(w, mag)
plt.xlabel("$\\omega$")
plt.ylabel("Magnitude")

Spot checking a few examples, we see it all checks out

In [ ]:
w = 2.5
n = np.arange(100)
a = 2*np.cos(w/2)
x = np.cos(w*n)
y = np.convolve(x, h)
plt.plot(y)
plt.plot([0, n.size], [a, a])
plt.plot([0, n.size], [-a, -a])

### General Causal Z-Transform

We can generalize the above expression for the convolution with any impulse response of length $L$

### Def. Convolution
### $y[n] = x*h[n] = \sum_{j = 0}^{len(h)-1} h[j] x[n-j]$


### $y[n] = h[0] e^{i \omega n} + h[1] e^{i \omega (n-1)} + h[2] e^{i \omega (n-2)} + .. $

### $y[n] = e^{i \omega n}(h[0]  + h[1] e^{-i \omega} + h[2] e^{-i 2 \omega} + ... ) $


### $y[n] = e^{i \omega n} \left( \sum_{k=0}^{L-1} h[k] (e^{-i \omega})^k  \right)$


In fact, the general expression relaxes the assumption that we're raising $e^{-i\omega}$ to successive powers and instead just plugs in some general complex number $z$

### $y[n] = e^{i \omega n} \left( \sum_{k=0}^{L-1} h[k] z^{-k}  \right)$

but we'll stick to the special case $z = e^{i \omega}$ for a lot of our examples

Let's define this in code below and apply it to a more complicated example that would be more annoying to do by hand:

In [ ]:
def z_transform(h, z):
    ret = 0
    for n in range(len(h)):
        ret += h[n]*(z**(-n))
    return ret

def get_z_over_unit_semicircle(h, n_samples=100):
    ws = np.linspace(0, np.pi, n_samples)
    ret = np.zeros(n_samples, dtype=complex)
    for i, w in enumerate(ws):
        ret[i] = z_transform(h, np.exp(1j*w))
    return ws, ret

plt.figure(figsize=(10, 5))
ramp = np.linspace(1, 0, 10)
plt.subplot(121)
plt.stem(ramp)
plt.subplot(122)
ws, ztransform = get_z_over_unit_semicircle(ramp, 100)
plt.plot(ws, np.abs(ztransform))
plt.xlabel("Radian Frequency")
plt.ylabel("Magnitude z-transform")

### Highpass Filter

Let's look at one more example analytically for now

### $h[0] = 1, h[1] = -1$

### $y[n] = x[n] - x[n-1]$

Why is this a highpass filter?  Intuation: suppose $y(t) = \sin(\omega t)$

then derivative

$y'(t) = \omega \cos( \omega t)$

the response is directly proportional to the frequency

### $y[n] = e^{i \omega n}(1 - e^{-i \omega}) \frac{ e^{i \omega/2} } {  e^{i \omega/2} }$

### $y[n] = e^{i \omega n} e^{-i \omega /2} (e^{i \omega/2} - e^{-i \omega / 2})$

### $y[n] = e^{i \omega n} i e^{-i \omega /2} (2 \sin (\omega/2))$

In [ ]:
w = np.linspace(0, np.pi, 100)
mag = 2*np.sin(w/2)
plt.plot(w, mag)
plt.xlabel("$\\omega$")
plt.ylabel("Magnitude")